# Homework 5: Comprehensive Data Cleaning Exercises

This document contains one theoretical exercise, definitions for data cleaning workflows, and three comprehensive practical exercises. The theoretical exercise will help you master the core concept of data leakage. Subsequently, the practical exercises will require you to apply the entire data cleaning process you have learned to specific datasets.

**Important Note**: The primary objective of these exercises is to practice **Data Cleaning** techniques. A Pipeline is a powerful tool that helps automate the process, making the code cleaner, more maintainable, and, most importantly, helping to prevent **data leakage**.

**Theoretical Exercise: Processing Workflow and Data Leakage**

Assume you have a dataset containing both missing values and outliers. Someone proposes the following processing workflow:

- **Step 1**: Remove outliers from the **entire** dataset.
- **Step 2**: Use SimpleImputer to fill in missing values on the **entire** dataset.
- **Step 3**: Split the cleaned dataset into a training set and a testing set.
- **Step 4**: Train a model on the training set.

**Answer the following questions:**

1. **Error Analysis**: Where did the process go wrong, and why does it cause data leakage?
2. **Consequences**: How will data leakage affect the model's evaluation metrics?
3. **The Correct Workflow**: Outline the correct sequence of steps to avoid data leakage.
4. **Role of the Pipeline**: How does a Scikit-learn Pipeline help solve this problem?

### 1. Error Analysis

The mistake lies in Steps 1 and 2:

- Removing outliers (Step 1) was done on the entire dataset before splitting into training and testing sets.

- Imputation (Step 2) was also done on the entire dataset before splitting.

=> Both steps allow information from the test set to “leak” into the training set.

When identifying outliers, the test set influences which rows in the training set are removed, whick leaking information.

Similarly, when imputing missing values, the imputer calculates statistics (mean, median, neighbors, regression relationships) using all data. This means the training process indirectly knows something about the distribution of the test set.

### 2. Consequences

If the data is leaked:

- The model will look artificially better in evaluation because it has indirectly seen patterns from the test set.

- Metrics such as accuracy, R², precision, recall, F1, AUC, etc. will be overly optimistic.

- In deployment, performance will drop sharply because the test-time conditions (where unseen data comes in with missing values and outliers) are no longer faithfully simulated.

### 3. The Correct Workflow

1. **Split the dataset** into training and testing sets.

2. **Outlier handling** (fit outlier detection only on training data):

    - Identify and remove outliers from X_train, y_train.

    - Leave the test set untouched (outliers are part of reality at test time).

3. **Preprocessing** (imputation, scaling, variance removal, etc.):

    - Fit imputers and transformers on X_train.

    - Apply the transformations to both X_train and X_test.

4. **Train the model** on the processed training data.

5. **Evaluate the model** on the transformed test set.

### 4. Role of the Pipeline:

Scikit-learn Pipelines help by:

- **Encapsulation**: Ensuring preprocessing (e.g., imputation, scaling, variance removal) is fit only on training data during .fit() and later applied to unseen data during .transform().

- **Leakage Prevention**: Pipelines prevent accidental use of the full dataset before splitting.

- **Reproducibility**: A single object encapsulates the workflow, ensuring transformations and model fitting are consistently applied.

- **Deployment readiness**: The exact training-time transformations are automatically applied to new, unseen data at inference.

In [3]:
import pandas as pd
import numpy as np
import io

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

### Practical Exercise 1: Oil Spill Dataset - Comparing the Effectiveness of a Comprehensive Cleaning Workflow

- **Dataset:** oil-spill.csv
- **Objective:** Apply and compare the effectiveness of a baseline pipeline versus a pipeline with comprehensive cleaning steps.

**Requirements:**

1. **Preparation:** Load the data, remove duplicates, and split it into training and testing sets.
2. **Compare two approaches:**
    - **Approach A:** Train Pipeline 1 (Baseline) on the original training set.
    - **Approach B:**
        1. Apply the Outlier Handling Workflow (IQR) on the training set.
        2. Train Pipeline 2 (Basic Cleaning) on the outlier-cleaned training set.
3. **Evaluation:** Calculate the accuracy of both approaches on the test set.
4. **Conclusion:** Create a comparison table and determine whether the comprehensive cleaning workflow improved model performance.

In [5]:
url_lab1 = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/oil-spill.csv'
df_lab1 = pd.read_csv(url_lab1, header=None)
df_lab1.head()

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,1,2558,1506.09,456.63,90,6395000.0,40.88,7.89,29780.0,0.19,...,2850.00,1000.00,763.16,135.46,3.73,0,33243.19,65.74,7.95,1
1,2,22325,79.11,841.03,180,55812500.0,51.11,1.21,61900.0,0.02,...,5750.00,11500.00,9593.48,1648.80,0.60,0,51572.04,65.73,6.26,0
2,3,115,1449.85,608.43,88,287500.0,40.42,7.34,3340.0,0.18,...,1400.00,250.00,150.00,45.13,9.33,1,31692.84,65.81,7.84,1
3,4,1201,1562.53,295.65,66,3002500.0,42.40,7.97,18030.0,0.19,...,6041.52,761.58,453.21,144.97,13.33,1,37696.21,65.67,8.07,1
4,5,312,950.27,440.86,37,780000.0,41.43,7.03,3350.0,0.17,...,1320.04,710.63,512.54,109.16,2.58,0,29038.17,65.66,7.35,0


In [6]:
df_lab1 = df_lab1.drop_duplicates()
print("Data shape after removing duplicates:", df_lab1.shape)

Data shape after removing duplicates: (937, 50)


In [7]:
X = df_lab1.iloc[:, :-1]
y = df_lab1.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, " Test set:", X_test.shape)

Training set: (749, 49)  Test set: (188, 49)


In [8]:
pipe1 = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", LogisticRegression(max_iter=1000))
])

pipe1.fit(X_train, y_train)
y_pred_a = pipe1.predict(X_test)
acc_a = accuracy_score(y_test, y_pred_a)

print("\nApproach A (Baseline) Accuracy:", acc_a)


Approach A (Baseline) Accuracy: 0.9574468085106383


C:\Users\ttrun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1

mask = ~((X_train < (Q1 - 1.5 * IQR)) | (X_train > (Q3 + 1.5 * IQR))).any(axis=1)

X_train_clean = X_train[mask] 
y_train_clean = y_train[mask]

print("Training set size before outlier removal:", len(X_train))
print("Training set size after outlier removal:", len(X_train_clean))

pipe2 = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("var_thresh", VarianceThreshold(threshold=0.05)),
    ("model", LogisticRegression(max_iter=1000))
])

pipe2.fit(X_train_clean, y_train_clean)
y_pred_b = pipe2.predict(X_test)
acc_b = accuracy_score(y_test, y_pred_b)

print("Approach B (Comprehensive Cleaning) Accuracy:", acc_b)

Training set size before outlier removal: 749
Training set size after outlier removal: 332
Approach B (Comprehensive Cleaning) Accuracy: 0.8723404255319149


C:\Users\ttrun\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
results = pd.DataFrame({
    "Approach": ["A. Baseline", "B. Comprehensive Cleaning"],
    "Accuracy": [acc_a, acc_b]
})
print("\nComparison of Approaches:")
print(results)


Comparison of Approaches:
                    Approach  Accuracy
0                A. Baseline  0.957447
1  B. Comprehensive Cleaning  0.872340


### Practical Exercise 2: Housing Dataset - Comparing Outlier Handling Methods

- **Dataset:** housing.csv
- **Objective:** Compare the effectiveness of different outlier handling methods in a regression problem.

**Requirements:**

1. **Preparation:** Load the data and split it into training/testing sets to predict MEDV.
2. **Compare four approaches:**
    - Approach A (Baseline): Train Pipeline 2 (Basic Cleaning) with a RandomForestRegressor model on the original training set.
    - Approach B (IQR): Apply the Outlier Handling Workflow (IQR), then train Pipeline 2 on the cleaned training set.
    - Approach C (Standard Deviation - Std): Apply the Outlier Handling Workflow (Std) with a threshold of 3 standard deviations, then train Pipeline 2 on the cleaned training set.
    - Approach D (LOF): Apply the Outlier Handling Workflow (LOF), then train Pipeline 2 on the cleaned training set.
3. **Evaluation:** Calculate the Mean Squared Error (MSE) for all four approaches on the test set.
4. **Conclusion:** Create a comparison table and comment on which outlier handling method was more effective for this dataset.

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [12]:
column_names = [
    "CRIM",    # Tội phạm bình quân đầu người theo thị trấn
    "ZN",      # Tỷ lệ đất ở > 25,000 sq.ft
    "INDUS",   # Tỷ lệ diện tích cho doanh nghiệp phi bán lẻ
    "CHAS",    # Biến giả sông Charles (=1 nếu gần sông, 0 nếu không)
    "NOX",     # Nồng độ oxit nitơ (phần triệu)
    "RM",      # Số phòng trung bình mỗi căn hộ
    "AGE",     # % căn hộ xây dựng trước 1940
    "DIS",     # Khoảng cách bình quân đến 5 trung tâm việc làm
    "RAD",     # Chỉ số khả năng tiếp cận đường cao tốc
    "TAX",     # Thuế bất động sản
    "PTRATIO", # Tỷ lệ học sinh/giáo viên
    "B",       # 1000(Bk - 0.63)^2, với Bk % dân da đen
    "LSTAT",   # % dân có địa vị kinh tế xã hội thấp
    "MEDV"     # Giá trị trung vị của nhà (ngàn USD)
]
url_lab2 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/housing.csv"
df_lab2 = pd.read_csv(url_lab2, header=None, names=column_names)
df_lab2
df_lab2.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [13]:
X = df_lab2.drop("MEDV", axis=1)
y = df_lab2["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
def build_pipeline():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])
    return pipeline

In [15]:
pipeA = build_pipeline()
pipeA.fit(X_train, y_train)
y_predA = pipeA.predict(X_test)
mseA = mean_squared_error(y_test, y_predA)

In [16]:
def remove_outliers_iqr(X, y):
    df = X.copy()
    df["MEDV"] = y
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    mask = ~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)
    return df.loc[mask].drop("MEDV", axis=1), df.loc[mask]["MEDV"]

X_train_iqr, y_train_iqr = remove_outliers_iqr(X_train, y_train)

pipeB = build_pipeline()
pipeB.fit(X_train_iqr, y_train_iqr)
y_predB = pipeB.predict(X_test)
mseB = mean_squared_error(y_test, y_predB)

In [17]:
def remove_outliers_std(X, y, threshold=3):
    df = X.copy()
    df["MEDV"] = y
    z_scores = ((df - df.mean()) / df.std()).abs()
    mask = (z_scores < threshold).all(axis=1)
    return df.loc[mask].drop("MEDV", axis=1), df.loc[mask]["MEDV"]

X_train_std, y_train_std = remove_outliers_std(X_train, y_train)

pipeC = build_pipeline()
pipeC.fit(X_train_std, y_train_std)
y_predC = pipeC.predict(X_test)
mseC = mean_squared_error(y_test, y_predC)

In [18]:
from sklearn.neighbors import LocalOutlierFactor

def remove_outliers_lof(X, y, contamination=0.05):
    lof = LocalOutlierFactor(contamination=contamination)
    yhat = lof.fit_predict(X)
    mask = yhat != -1
    return X[mask], y[mask]

X_train_lof, y_train_lof = remove_outliers_lof(X_train, y_train)

pipeD = build_pipeline()
pipeD.fit(X_train_lof, y_train_lof)
y_predD = pipeD.predict(X_test)
mseD = mean_squared_error(y_test, y_predD)

In [19]:
results = pd.DataFrame({
    "Approach": ["Baseline", "IQR", "Std (3σ)", "LOF"],
    "MSE": [mseA, mseB, mseC, mseD]
})
print(results)

   Approach        MSE
0  Baseline   7.912745
1       IQR  26.749293
2  Std (3σ)  12.881738
3       LOF   9.002562


### Practical Exercise 3: Horse Colic Dataset - Comparing Imputation Methods

- **Dataset:** horse-colic.csv
- **Objective:** Compare different imputation methods after performing standard cleaning steps.

**Requirements:**

1. **Preparation:** Load the data and split it into training and testing sets.
2. **Basic Cleaning:** Apply the Outlier Handling Workflow (IQR) on the training set.
3. **Compare three Pipelines:**
    - Approach A: Train Pipeline 2 (Basic Cleaning) with SimpleImputer(strategy='median').
    - Approach B: Train Pipeline 3a (KNN Imputation).
    - Approach C: Train Pipeline 3b (Iterative Imputation).
    - Note: All pipelines are to be trained on the outlier-cleaned training set.
4. **Evaluation:** Use accuracy_score and classification_report to evaluate all three approaches on the test set.
5. **Conclusion:** Create a comparison table and hypothesize why one imputation method might be more effective than others for this particular dataset.


In [20]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [21]:
url_lab3 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/horse-colic.csv"

col_names = [
    "surgery", "age", "hospital_number", "rectal_temp", "pulse",
    "respiratory_rate", "temp_extremities", "peripheral_pulse",
    "mucous_membrane", "capillary_refill", "pain", "peristalsis",
    "abdominal_distension", "nasogastric_tube", "nasogastric_reflux",
    "nasogastric_reflux_ph", "rectal_exam_feces", "abdomen",
    "packed_cell_volume", "total_protein", "abdomocentesis_appearance",
    "abdomocentesis_total_protein", "outcome", "surgical_lesion",
    "lesion_1", "lesion_2", "lesion_3", "lesion_4"
]

df_lab3 = pd.read_csv(url_lab3, header=None, names=col_names, na_values="?")

df_lab3.head()

,surgery,age,hospital_number,rectal_temp,pulse,respiratory_rate,temp_extremities,peripheral_pulse,mucous_membrane,capillary_refill,...,packed_cell_volume,total_protein,abdomocentesis_appearance,abdomocentesis_total_protein,outcome,surgical_lesion,lesion_1,lesion_2,lesion_3,lesion_4
0,2.0,1,530101,38.5,66.0,28.0,3.0,3.0,NaN,2.0,...,45.0,8.4,NaN,NaN,2.0,2,11300,0,0,2
1,1.0,1,534817,39.2,88.0,20.0,NaN,NaN,4.0,1.0,...,50.0,85.0,2.0,2.0,3.0,2,2208,0,0,2
2,2.0,1,530334,38.3,40.0,24.0,1.0,1.0,3.0,1.0,...,33.0,6.7,NaN,NaN,1.0,2,0,0,0,1
3,1.0,9,5290409,39.1,164.0,84.0,4.0,1.0,6.0,2.0,...,48.0,7.2,3.0,5.3,2.0,1,2208,0,0,1
4,2.0,1,530255,37.3,104.0,35.0,NaN,NaN,6.0,2.0,...,74.0,7.4,NaN,NaN,2.0,2,4300,0,0,2


In [22]:
df_lab3 = df_lab3.dropna(subset=["outcome"])

X = df_lab3.drop("outcome", axis=1)
y = df_lab3["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [23]:
def remove_outliers_iqr(X, y):
    df = X.copy()
    df["outcome"] = y
    num_cols = df.select_dtypes(include=[np.number]).columns
    Q1 = df[num_cols].quantile(0.25)
    Q3 = df[num_cols].quantile(0.75)
    IQR = Q3 - Q1
    mask = ~((df[num_cols] < (Q1 - 1.5 * IQR)) | (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)
    return df.loc[mask].drop("outcome", axis=1), df.loc[mask]["outcome"]

X_train_clean, y_train_clean = remove_outliers_iqr(X_train, y_train)

In [24]:
pipeA = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

In [25]:
pipeB = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

In [26]:
pipeC = Pipeline([
    ("imputer", IterativeImputer(random_state=42)),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

In [27]:
pipelines = {"Median": pipeA, "KNN": pipeB, "Iterative": pipeC}
results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train_clean, y_train_clean)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append([name, acc])
    print(f"\n{name} Imputation:")
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results, columns=["Approach", "Accuracy"])
print(results_df)


Median Imputation:
              precision    recall  f1-score   support

         1.0       0.77      0.83      0.80        36
         2.0       0.73      0.73      0.73        15
         3.0       0.17      0.11      0.13         9

    accuracy                           0.70        60
   macro avg       0.56      0.56      0.56        60
weighted avg       0.67      0.70      0.68        60


KNN Imputation:
              precision    recall  f1-score   support

         1.0       0.76      0.78      0.77        36
         2.0       0.76      0.87      0.81        15
         3.0       0.17      0.11      0.13         9

    accuracy                           0.70        60
   macro avg       0.56      0.59      0.57        60
weighted avg       0.67      0.70      0.68        60


Iterative Imputation:
              precision    recall  f1-score   support

         1.0       0.74      0.78      0.76        36
         2.0       0.79      0.73      0.76        15
         3.0   